# Cibuco_Boriken — BirdCLEF+ 2026 Inference v7 (Perch Native)
**Team:** Cibuco_Boriken | **Model:** Google Perch (native classifier) | **234 species**

Uses Perch's **built-in species classifier** (trained on all of Xeno-Canto) instead of our custom MLP head.
Perch can predict 203/234 competition species natively.

Reference: https://www.kaggle.com/models/google/bird-vocalization-classifier

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q librosa audioread tensorflow
print('Dependencies installed ✅')

In [ ]:
# ── Cell 2: Imports + config ──
import os
import json
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# Paths
DATA_DIR = Path('/kaggle/input/competitions/birdclef-2026')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

# ⬇️ UPDATE THIS: path to the Perch model from Kaggle Models
# Attach: google/bird-vocalization-classifier → look for the SavedModel dir
PERCH_MODEL_DIR = Path('/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1')

# Config
SAMPLE_RATE = 32000
WINDOW_SEC  = 5.0

ZERO_SHOT_PRIOR = 1.0 / 234

print(f'Perch model dir: {PERCH_MODEL_DIR}')
print(f'Exists: {PERCH_MODEL_DIR.exists()}')
print('Config ready ✅')

In [ ]:
# ── Cell 3: Discover Perch model structure ──
# List all files in the model directory to find label files
print('=== Model directory contents ===')
for root, dirs, files in os.walk(str(PERCH_MODEL_DIR.parent.parent)):
    level = root.replace(str(PERCH_MODEL_DIR.parent.parent), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = ' ' * 2 * (level + 1)
    for f in files[:20]:  # limit per dir
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        print(f'{sub_indent}{f}  ({size:,} bytes)')
    if len(files) > 20:
        print(f'{sub_indent}... and {len(files)-20} more files')

In [ ]:
# ── Cell 4: Load Perch model + discover outputs ──
perch = tf.saved_model.load(str(PERCH_MODEL_DIR))
print('Perch model loaded ✅')

# Discover callable signatures
print('\n=== Model signatures ===')
if hasattr(perch, 'signatures'):
    for key in perch.signatures:
        print(f'  Signature: {key}')
        sig = perch.signatures[key]
        print(f'    Inputs:  {sig.structured_input_signature}')
        print(f'    Outputs: {sig.structured_outputs}')

# Check available methods
print('\n=== Available methods ===')
for attr in dir(perch):
    if not attr.startswith('_'):
        print(f'  {attr}: {type(getattr(perch, attr))}')

In [ ]:
# ── Cell 5: Test inference on silence to see output shapes ──
test_audio = np.zeros(int(SAMPLE_RATE * WINDOW_SEC), dtype=np.float32)
test_input = tf.constant(test_audio[np.newaxis, :], dtype=tf.float32)

# Try different calling methods
output = None
for method_name in ['infer_tf', 'front_end', '__call__']:
    try:
        if method_name == '__call__':
            out = perch(test_input)
        else:
            fn = getattr(perch, method_name, None)
            if fn is None:
                continue
            out = fn(test_input)
        output = out
        print(f'Method: {method_name} ✅')
        break
    except Exception as e:
        print(f'Method: {method_name} ❌ ({e})')

if output is not None:
    if isinstance(output, dict):
        print('\nOutput type: dict')
        for k, v in output.items():
            print(f'  {k}: shape={v.shape}, dtype={v.dtype}')
    elif isinstance(output, (tuple, list)):
        print(f'\nOutput type: tuple/list with {len(output)} elements')
        for i, v in enumerate(output):
            print(f'  [{i}]: shape={v.shape}, dtype={v.dtype}')
    else:
        print(f'\nOutput type: {type(output)}, shape={output.shape}')
else:
    print('\n⚠️ Could not call model — check signatures above')

In [ ]:
# ── Cell 6: Find species labels ──
# Perch typically stores labels in the model assets or as a separate file
import glob

# Search for label files
label_candidates = []
search_root = str(PERCH_MODEL_DIR.parent.parent)
for pattern in ['**/*label*', '**/*class*', '**/*species*', '**/*.csv', '**/*.txt', '**/*.json']:
    label_candidates.extend(glob.glob(os.path.join(search_root, pattern), recursive=True))

# Also check inside SavedModel assets
assets_dir = PERCH_MODEL_DIR / 'assets'
if assets_dir.exists():
    for f in assets_dir.iterdir():
        label_candidates.append(str(f))

label_candidates = sorted(set(label_candidates))
print(f'Found {len(label_candidates)} potential label files:')
for f in label_candidates:
    size = os.path.getsize(f)
    print(f'  {f}  ({size:,} bytes)')
    # Print first few lines of small files
    if size < 50000:
        try:
            with open(f, 'r') as fh:
                lines = fh.readlines()[:10]
            print(f'    First lines: {[l.strip() for l in lines]}')
        except:
            pass

In [ ]:
# ── Cell 7: Build species mapping ──
# Perch v2 has two label files:
#   - labels.csv (14,795 entries — full Perch label set, index matches model output)
#   - perch_v2_ebird_classes.csv (eBird codes subset)
# We use labels.csv as the canonical index→species mapping.

taxonomy_df = pd.read_csv(DATA_DIR / 'taxonomy.csv')
SPECIES = taxonomy_df['primary_label'].tolist()
NUM_SPECIES = len(SPECIES)

# Load Perch label index (14,795 species — matches model 'label' output dim)
label_file = PERCH_MODEL_DIR / 'assets' / 'labels.csv'
label_df = pd.read_csv(label_file)
print(f'Label file: {label_file}')
print(f'Shape: {label_df.shape}, Columns: {label_df.columns.tolist()}')
print(label_df.head())

# Detect the eBird code column
ebird_col = None
for col in ['ebird2021', 'ebird_code', 'species_code', label_df.columns[0]]:
    if col in label_df.columns:
        ebird_col = col
        break
if ebird_col is None:
    ebird_col = label_df.columns[0]
print(f'\nUsing column: {ebird_col}')

perch_labels = label_df[ebird_col].tolist()
print(f'Perch knows {len(perch_labels)} species')

# Map Perch label index → competition species
perch_to_idx = {sp: i for i, sp in enumerate(perch_labels)}
mapped = [sp for sp in SPECIES if sp in perch_to_idx]
unmapped = [sp for sp in SPECIES if sp not in perch_to_idx]
print(f'Mapped to competition: {len(mapped)}/{NUM_SPECIES}')
print(f'Unmapped (zero-shot):  {len(unmapped)}')
if unmapped:
    print(f'Unmapped species: {unmapped}')

In [ ]:
# ── Cell 8: Parse sample submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'Sample submission: {len(sample_sub)} rows, {len(sample_sub.columns)} cols')

def parse_row_id(row_id):
    parts = row_id.rsplit('_', 1)
    return parts[0], int(parts[1])

file_windows = defaultdict(list)
for row_id in sample_sub['row_id']:
    stem, end_time = parse_row_id(row_id)
    file_windows[stem].append((row_id, end_time))

print(f'Soundscape files: {len(file_windows)}')
print('Row IDs parsed ✅')

In [ ]:
# ── Cell 9: Perch native inference ──
import time

soundscape_dir = DATA_DIR / 'test_soundscapes'
window_samples = int(WINDOW_SEC * SAMPLE_RATE)

# Get the serving function (much faster than calling perch() directly)
infer_fn = perch.signatures['serving_default']

all_rows = {}
t_start = time.time()
total_windows = sum(len(v) for v in file_windows.values())
processed = 0

for file_stem in tqdm(sorted(file_windows.keys()), desc='Perch Native'):
    # Find audio file
    audio_path = None
    for ext in ['.ogg', '.wav', '.flac']:
        candidate = soundscape_dir / f'{file_stem}{ext}'
        if candidate.exists():
            audio_path = candidate
            break

    if audio_path is None:
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    try:
        audio, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f'Error loading {audio_path}: {e}')
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    for row_id, end_time in file_windows[file_stem]:
        start_sample = max(0, int((end_time - WINDOW_SEC) * SAMPLE_RATE))
        end_sample = int(end_time * SAMPLE_RATE)
        chunk = audio[start_sample:end_sample]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))

        # Run Perch via serving_default signature
        waveform = tf.constant(chunk[np.newaxis, :], dtype=tf.float32)
        output = infer_fn(inputs=waveform)

        # 'label' output has shape (1, 14795) — raw logits for all Perch species
        logits = output['label'].numpy().squeeze()  # (14795,)

        # Convert logits to probabilities
        probs = 1.0 / (1.0 + np.exp(-logits))  # sigmoid

        # Build row using species mapping
        row = {'row_id': row_id}
        for sp in SPECIES:
            if sp in perch_to_idx:
                row[sp] = float(probs[perch_to_idx[sp]])
            else:
                row[sp] = ZERO_SHOT_PRIOR
        all_rows[row_id] = row

        processed += 1
        if processed % 100 == 0:
            elapsed = time.time() - t_start
            rate = processed / elapsed
            remaining = (total_windows - processed) / rate / 60
            print(f'  {processed}/{total_windows} windows | {elapsed/60:.1f} min elapsed | ~{remaining:.1f} min remaining')

elapsed = time.time() - t_start
print(f'\nTotal rows: {len(all_rows)} (expected: {len(sample_sub)})')
print(f'Total time: {elapsed/60:.1f} min ✅')

In [ ]:
# ── Cell 10: Generate submission.csv ──
rows_ordered = [all_rows[rid] for rid in sample_sub['row_id']]
submission_df = pd.DataFrame(rows_ordered)
submission_df = submission_df[sample_sub.columns]
submission_df = submission_df.fillna(ZERO_SHOT_PRIOR)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Submission saved ✅')
print(f'Shape: {submission_df.shape} (expected: {sample_sub.shape})')
print(f'Min prob: {submission_df.iloc[:, 1:].min().min():.6f}')
print(f'Max prob: {submission_df.iloc[:, 1:].max().max():.6f}')
print(submission_df.head(3))

In [ ]:
# ── Cell 11: Validate submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
our_sub    = pd.read_csv(OUTPUT_PATH)

print('=== Submission Validation ===')
print(f'Expected rows:  {len(sample_sub)}')
print(f'Our rows:       {len(our_sub)}')
print(f'Expected cols:  {len(sample_sub.columns)}')
print(f'Our cols:       {len(our_sub.columns)}')
print(f'Columns match:  {set(sample_sub.columns) == set(our_sub.columns)}')
print(f'Row IDs match:  {set(sample_sub["row_id"]) == set(our_sub["row_id"])}')
print(f'No NaN:         {our_sub.isna().sum().sum() == 0}')
print(f'No zeros:       {(our_sub.iloc[:, 1:] == 0).sum().sum() == 0}')
print()
if set(sample_sub.columns) == set(our_sub.columns):
    print('✅ PERCH NATIVE SUBMISSION VALID — ready to submit!')
else:
    missing = set(sample_sub.columns) - set(our_sub.columns)
    extra   = set(our_sub.columns) - set(sample_sub.columns)
    if missing: print(f'⚠️  Missing columns: {missing}')
    if extra:   print(f'⚠️  Extra columns: {extra}')